# Schedule Health Readiness Gate v1

本实验验证固定 `schedule_health.json` artifact 是否能进入 dashboard export，并作为 readiness overview 的正式 gate。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / 'AGENTS.md').exists() and (path / 'src').exists()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import build_dashboard_view_model, write_dashboard_html, write_model_matrix_dashboard_export, write_schedule_health
from src.schedule import JobResult

EXPERIMENT = 'schedule_health_readiness_gate_v1'
CARD = REPO_ROOT / 'research' / EXPERIMENT
SOURCE_CARD = REPO_ROOT / 'research/analysis_segment_effect_matrix_v1'
SCHEDULE_HEALTH = CARD / 'schedule_health.json'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
DASHBOARD_HTML = CARD / 'dashboard.html'
SUMMARY_PATH = CARD / 'analysis_summary.json'
CARD.mkdir(parents=True, exist_ok=True)

## 写出固定 schedule health artifact

用一个成功的数据更新 job 和一个失败的 dashboard refresh job 构造真实多 job 汇总。

In [2]:
results = [
    JobResult(
        name='data_update',
        status='success',
        payload={
            'attempts': 1,
            'retry_schema_version': 'schedule.retry_attempts.v1',
            'retry_error_count': 0,
            'retry_errors': [],
        },
    ),
    JobResult(
        name='model_dashboard_refresh',
        status='failed',
        payload={
            'attempts': 2,
            'retry_schema_version': 'schedule.retry_attempts.v1',
            'retry_error_count': 2,
            'retry_errors': [
                {'attempt': 1, 'error_type': 'RuntimeError', 'message': 'temporary dashboard failure'},
                {'attempt': 2, 'error_type': 'RuntimeError', 'message': 'permanent dashboard failure'},
            ],
        },
        detail='permanent dashboard failure',
    ),
]

schedule_health = write_schedule_health(results, output_path=SCHEDULE_HEALTH)
assert schedule_health['schema_version'] == 'dashboard.schedule_health.v1'
assert schedule_health['job_count'] == 2
assert schedule_health['failed_job_count'] == 1
assert schedule_health['total_retry_error_count'] == 2
assert schedule_health['top_failure_job'] == 'model_dashboard_refresh'
assert schedule_health['top_failure_last_error'] == 'RuntimeError:permanent dashboard failure'
schedule_health

{'schema_version': 'dashboard.schedule_health.v1',
 'job_count': 2,
 'failed_job_count': 1,
 'total_attempts': 3,
 'total_retry_error_count': 2,
 'top_failure_job': 'model_dashboard_refresh',
 'top_failure_last_error': 'RuntimeError:permanent dashboard failure',
 'jobs': [{'name': 'data_update',
   'status': 'success',
   'attempts': 1,
   'retry_schema_version': 'schedule.retry_attempts.v1',
   'retry_error_count': 0,
   'last_error_attempt': 0,
   'last_error_type': '',
   'last_error_message': '',
   'detail': ''},
  {'name': 'model_dashboard_refresh',
   'status': 'failed',
   'attempts': 2,
   'retry_schema_version': 'schedule.retry_attempts.v1',
   'retry_error_count': 2,
   'last_error_attempt': 2,
   'last_error_type': 'RuntimeError',
   'last_error_message': 'permanent dashboard failure',
   'detail': 'permanent dashboard failure'}]}

## 接入 dashboard readiness overview

dashboard export 明确读取 `schedule_health_path`，view model 不读取 sidecar 路径，只消费 export 内联 payload。

In [3]:
dashboard_payload = write_model_matrix_dashboard_export(
    SOURCE_CARD,
    output_path=DASHBOARD_EXPORT,
    top_n=3,
    schedule_health_path=SCHEDULE_HEALTH,
)
view_model = build_dashboard_view_model(dashboard_payload)
html = write_dashboard_html(view_model, output_path=DASHBOARD_HTML, title='SOXL Schedule Health Readiness')

overview = view_model['readiness_overview']
gates = {gate['gate']: gate for gate in overview['gates']}
assert dashboard_payload['schedule_health'] == schedule_health
assert 'artifacts' not in view_model
assert overview['schema_version'] == 'dashboard.readiness_overview.v1'
assert overview['status'] == 'blocked'
assert gates['dashboard_manifest']['status'] == 'pass'
assert gates['schedule_health']['status'] == 'block'
assert gates['schedule_health']['detail'] == 'failed 1 / jobs 2 / retry_errors 2 / top model_dashboard_refresh RuntimeError:permanent dashboard failure'
assert 'Schedule Health' in html
assert 'schedule_health.json' not in html
overview

{'schema_version': 'dashboard.readiness_overview.v1',
 'status': 'blocked',
 'gate_count': 3,
 'pass_count': 1,
 'review_count': 1,
 'block_count': 1,
 'gates': [{'gate': 'dashboard_manifest',
   'label': 'Dashboard Manifest',
   'status': 'pass',
   'detail': 'analysis_manifest / complete hash b84fe5b400',
   'source_schema_version': 'artifact.v1'},
  {'gate': 'schedule_health',
   'label': 'Schedule Health',
   'status': 'block',
   'detail': 'failed 1 / jobs 2 / retry_errors 2 / top model_dashboard_refresh RuntimeError:permanent dashboard failure',
   'source_schema_version': 'dashboard.schedule_health.v1'},
  {'gate': 'model_runs',
   'label': 'Model Runs',
   'status': 'review',
   'detail': 'trained 0 / backtested 0 / expected 0',
   'source_schema_version': 'model.run_ledger_summary.v1'}]}

## 汇总 sidecar

记录本次 readiness gate 的状态，便于 dashboard、schedule 和 production review 对齐。

In [4]:
summary = {
    'schema_version': 'research.schedule_health_readiness_gate_v1.summary.v1',
    'experiment': EXPERIMENT,
    'schedule_health_schema_version': schedule_health['schema_version'],
    'job_count': schedule_health['job_count'],
    'failed_job_count': schedule_health['failed_job_count'],
    'total_retry_error_count': schedule_health['total_retry_error_count'],
    'top_failure_job': schedule_health['top_failure_job'],
    'top_failure_last_error': schedule_health['top_failure_last_error'],
    'readiness_overview_schema_version': overview['schema_version'],
    'readiness_overview_status': overview['status'],
    'readiness_overview_gate_count': overview['gate_count'],
    'readiness_overview_block_gates': ','.join(gate['gate'] for gate in overview['gates'] if gate['status'] == 'block'),
    'dashboard_export': str(DASHBOARD_EXPORT.relative_to(REPO_ROOT)),
    'dashboard_html': str(DASHBOARD_HTML.relative_to(REPO_ROOT)),
    'promotion_decision': 'promote_to_src_core_readiness_and_src_dashboard_schedule_health',
    'risk_note': 'schedule health gate 只反映最近收集到的 JobResult，不替代 broker/provider 错误分类。',
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.schedule_health_readiness_gate_v1.summary.v1',
 'experiment': 'schedule_health_readiness_gate_v1',
 'schedule_health_schema_version': 'dashboard.schedule_health.v1',
 'job_count': 2,
 'failed_job_count': 1,
 'total_retry_error_count': 2,
 'top_failure_job': 'model_dashboard_refresh',
 'top_failure_last_error': 'RuntimeError:permanent dashboard failure',
 'readiness_overview_schema_version': 'dashboard.readiness_overview.v1',
 'readiness_overview_status': 'blocked',
 'readiness_overview_gate_count': 3,
 'readiness_overview_block_gates': 'schedule_health',
 'dashboard_export': 'research/schedule_health_readiness_gate_v1/dashboard_export.json',
 'dashboard_html': 'research/schedule_health_readiness_gate_v1/dashboard.html',
 'promotion_decision': 'promote_to_src_core_readiness_and_src_dashboard_schedule_health',
 'risk_note': 'schedule health gate 只反映最近收集到的 JobResult，不替代 broker/provider 错误分类。'}

## 中文结论

- 做了什么：新增固定 `schedule_health.json` 写出契约，并把 `schedule_health` 纳入 dashboard readiness overview 的正式 gate。
- 核心指标：本实验包含 2 个 schedule job，1 个失败 job，`total_retry_error_count=2`；readiness overview 状态为 `blocked`，block gate 为 `schedule_health`。
- regime 差异：这里是运行状态 regime；成功 job 通过，失败 job 的 attempts、retry_errors 和 last_error 会触发 schedule health gate block。
- 是否晋升：晋升到 `src/dashboard.write_schedule_health` 和 `src/core.build_dashboard_readiness_overview`，因为 dashboard、schedule 和 production review 都需要统一判断运行健康状态。
- 风险：该 gate 只能反映最近写入的 schedule health artifact；如果 scheduler 没有定期刷新该 artifact，dashboard 可能展示过期运行状态。
- 下一步：把 production/paper run 的实际 data update、dashboard refresh、paper execution 结果统一收集后写入 production 目录下的 schedule health artifact。